# 🚦 GridLock V7 (Leak Fixed) — CatBoost + LightGBM
**Strategy:** 5-fold CV, 3000-3500 iterations, rich time + geohash features, no day-derived features.

In [ ]:
import warnings, os, gc, time
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
    print("✅ CatBoost available")
except ImportError:
    HAS_CATBOOST = False
    print("⚠️ CatBoost not available — will use extra LGB")
print("✅ All imports successful")


In [ ]:
def find_file(name):
    for root in [Path('/kaggle/input'), Path('/kaggle/working'), Path('dataset'), Path('.')]:
        if root.exists():
            m = sorted(root.rglob(name))
            if m: return m[0]
    raise FileNotFoundError(name)

train = pd.read_csv(find_file('train.csv'))
test  = pd.read_csv(find_file('test.csv'))
try:
    sample_sub = pd.read_csv(find_file('sample_submission.csv'))
except:
    sample_sub = None

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Train days: {train['day'].min()}-{train['day'].max()}")
print(f"Test  days: {test['day'].min()}-{test['day'].max()}")
print(f"Train slots: {train['timestamp'].iloc[0]} → {train['timestamp'].iloc[-1]}")
print(f"Test  slots: {test['timestamp'].iloc[0]} → {test['timestamp'].iloc[-1]}")
print(f"Target: mean={train['demand'].mean():.5f}, std={train['demand'].std():.5f}")


In [ ]:
def engineer_features(df):
    df = df.copy()

    # ── Time features ──
    df['hour']      = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']    = df['timestamp'].str.split(':').str[1].astype(int)
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15

    # Cyclic encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']      / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']      / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

    # Peak flags
    df['is_morning_peak'] = df['hour'].between(7, 9).astype(int)
    df['is_evening_peak'] = df['hour'].between(17, 19).astype(int)
    df['is_night']   = (df['hour'] < 6).astype(int)
    df['is_midday']  = df['hour'].between(11, 14).astype(int)
    df['is_rush']    = (df['is_morning_peak'] | df['is_evening_peak']).astype(int)

    # ── Geohash features ──
    df['geo3'] = df['geohash'].str[:3]
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]
    df['geo_len'] = df['geohash'].str.len()

    # ── Road / infrastructure ──
    df['RoadType']      = df['RoadType'].fillna('Unknown').astype(str)
    df['LargeVehicles'] = (df['LargeVehicles'].astype(str).str.strip() == 'Allowed').astype(int)
    df['Landmarks']     = (df['Landmarks'].astype(str).str.strip() == 'Yes').astype(int)
    df['NumberofLanes'] = pd.to_numeric(df['NumberofLanes'], errors='coerce').fillna(0).astype(int)
    df['capacity_score'] = df['NumberofLanes'] * (1 + df['LargeVehicles'])
    df['lanes_road_interact'] = df['NumberofLanes'].astype(str) + '_' + df['RoadType']

    # ── Weather ──
    df['Weather']     = df['Weather'].fillna('Unknown').astype(str)
    df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')
    global_temp_mean  = df['Temperature'].mean()
    df['Temperature'] = df['Temperature'].fillna(global_temp_mean)
    df['temp_bucket'] = pd.cut(df['Temperature'], bins=[-999,0,10,20,30,999],
                                labels=[0,1,2,3,4]).astype(float).fillna(2).astype(int)

    # ── Geo × time interactions ──
    df['geo4_hour'] = df['geo4'] + '_h' + df['hour'].astype(str)
    df['geo4_slot'] = df['geo4'] + '_s' + df['time_slot'].astype(str)
    df['geo5_slot'] = df['geo5'] + '_s' + df['time_slot'].astype(str)
    df['weather_slot'] = df['Weather'] + '_s' + df['time_slot'].astype(str)
    df['road_slot']  = df['RoadType'] + '_s' + df['time_slot'].astype(str)

    return df

print("Engineering features...")
train = engineer_features(train)
test  = engineer_features(test)
print(f"Train: {train.shape}, Test: {test.shape}")


In [ ]:
# ── Geohash frequency (combined train+test, no leakage) ──
combined_geo = pd.concat([train['geohash'], test['geohash']], ignore_index=True)
geo_freq = combined_geo.value_counts().to_dict()
train['geohash_freq'] = train['geohash'].map(geo_freq)
test['geohash_freq']  = test['geohash'].map(geo_freq)
print(f"Geohash freq: {len(geo_freq)} unique, min={min(geo_freq.values())}, max={max(geo_freq.values())}")

global_mean = train['demand'].mean()

# ── Slot-level global demand mean (Safe target encoding per time_slot) ──
# We use KFold to prevent leakage for time_slot mean
oof_slot_te = np.zeros(len(train))
from sklearn.model_selection import KFold
kf_slot = KFold(n_splits=5, shuffle=True, random_state=42)
for tri, vali in kf_slot.split(train):
    sm = train.iloc[tri].groupby('time_slot')['demand'].mean()
    oof_slot_te[vali] = train.iloc[vali]['time_slot'].map(sm).fillna(global_mean)
train['slot_demand_mean'] = oof_slot_te

full_slot_mean = train.groupby('time_slot')['demand'].mean()
test['slot_demand_mean'] = test['time_slot'].map(full_slot_mean).fillna(global_mean)

# ── Lag-96: same geohash, previous day ──
train['_abs'] = train['day'] * 96 + train['time_slot']
test['_abs']  = test['day']  * 96 + test['time_slot']
lag_lkp = (train.groupby(['geohash','_abs'])['demand']
           .mean().reset_index().rename(columns={'demand':'lag96'}))
lag_lkp['_abs'] = lag_lkp['_abs'] + 96  # EXACTLY 1 DAY SHIFT
train = train.merge(lag_lkp, on=['geohash','_abs'], how='left')
test  = test.merge(lag_lkp, on=['geohash','_abs'], how='left')
train['lag96_miss'] = train['lag96'].isna().astype(int)
test['lag96_miss']  = test['lag96'].isna().astype(int)
train['lag96'] = train['lag96'].fillna(global_mean)
test['lag96']  = test['lag96'].fillna(global_mean)
train.drop(columns=['_abs'], inplace=True)
test.drop(columns=['_abs'], inplace=True)
print(f"Lag-96 test missing: {test['lag96_miss'].sum()}/{len(test)} ({100*test['lag96_miss'].mean():.1f}%)")
print(f"After feature build — Train: {train.shape}, Test: {test.shape}")


In [ ]:
# ── OOF-safe target encoding ──
TARGET, SMOOTHING, N_SPLITS, SEED = 'demand', 30, 5, 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

TE_COLS = ['geohash','geo3','geo4','geo5',
           'RoadType','Weather',
           'geo4_hour','geo4_slot','geo5_slot',
           'lanes_road_interact','weather_slot','road_slot']

def target_encode_cv(tr, te, col):
    gm = tr[TARGET].mean()
    oof_te = np.zeros(len(tr))
    for tri, vali in kf.split(tr):
        stats = tr.iloc[tri].groupby(col)[TARGET].agg(['mean','count'])
        stats['smooth'] = (stats['mean']*stats['count'] + gm*SMOOTHING) / (stats['count']+SMOOTHING)
        oof_te[vali] = tr.iloc[vali][col].map(stats['smooth']).fillna(gm)
    fs = tr.groupby(col)[TARGET].agg(['mean','count'])
    fs['smooth'] = (fs['mean']*fs['count'] + gm*SMOOTHING) / (fs['count']+SMOOTHING)
    te_enc = te[col].map(fs['smooth']).fillna(gm)
    return oof_te, te_enc.values

for col in TE_COLS:
    tr_enc, te_enc = target_encode_cv(train, test, col)
    train[col+'_te'] = tr_enc
    test[col+'_te']  = te_enc
    print(f"  TE: {col}")
print(f"Target encoding done ✅ ({len(TE_COLS)} cols)")


In [ ]:
# ── Label encode raw categoricals ──
CAT_LABEL = ['geohash','geo3','geo4','geo5',
             'RoadType','Weather',
             'geo4_hour','geo4_slot','geo5_slot',
             'lanes_road_interact','weather_slot','road_slot']

combined = pd.concat([train, test], ignore_index=True)
for col in CAT_LABEL:
    if combined[col].dtype == object:
        combined[col] = LabelEncoder().fit_transform(combined[col].astype(str))
train_n = len(train)
train = combined.iloc[:train_n].copy()
test  = combined.iloc[train_n:].copy()

# ── Define feature set — NEVER include 'day' ──
DROP_COLS = {'Index','demand','timestamp','day'}
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]
X_train = train[FEATURE_COLS].fillna(-999)
y_train = train[TARGET]
X_test  = test[FEATURE_COLS].fillna(-999)
print(f"Features: {len(FEATURE_COLS)}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
assert 'day' not in FEATURE_COLS, "day leaked into features!"


In [ ]:
# ── Cross-validation runner ──
def run_kfold(model, X, y, X_te, label='Model', use_log=True):
    oof = np.zeros(len(X))
    preds = np.zeros(len(X_te))
    scores = []
    _y = np.log1p(y.values) if use_log else y.values
    for fold, (tri, vali) in enumerate(kf.split(X), 1):
        model.fit(X.iloc[tri], _y[tri])
        vp = model.predict(X.iloc[vali])
        tp = model.predict(X_te)
        if use_log:
            vp = np.expm1(vp); tp = np.expm1(tp)
            yv = y.values[vali]
        else:
            yv = y.values[vali]
        vp = np.clip(vp, 0, None); tp = np.clip(tp, 0, None)
        r2 = r2_score(yv, vp)
        scores.append(r2); oof[vali] = vp; preds += tp / N_SPLITS
        print(f"  Fold {fold}: R²={r2:.5f}")
    print(f"[{label}] Mean R²={np.mean(scores):.5f} ± {np.std(scores):.5f}\n")
    return oof, preds, scores


In [ ]:
# ── MODEL 1: LightGBM (Optuna #29, 3500 iters) ──
print("=== LightGBM (3500 iters) ===")
lgb_p = dict(objective='regression_l1', metric='rmse',
    learning_rate=0.04, n_estimators=3500, num_leaves=239,
    max_depth=-1, min_child_samples=5,
    subsample=0.7957, colsample_bytree=0.7169,
    reg_alpha=0.006055, reg_lambda=1.2769,
    n_jobs=-1, random_state=SEED, verbose=-1)
oof_lgb, pred_lgb, _ = run_kfold(lgb.LGBMRegressor(**lgb_p), X_train, y_train, X_test, 'LGB-3500', use_log=True)


In [ ]:
# ── MODEL 2: LightGBM seed-2 (3500 iters, different cols) ──
print("=== LightGBM Seed-2 ===")
lgb_p2 = dict(lgb_p); lgb_p2.update({'random_state':SEED+7,'colsample_bytree':0.65,'num_leaves':200})
oof_lgb2, pred_lgb2, _ = run_kfold(lgb.LGBMRegressor(**lgb_p2), X_train, y_train, X_test, 'LGB-S2', use_log=True)


In [ ]:
# ── MODEL 3: CatBoost (3000-3500 iters) ──
if HAS_CATBOOST:
    print("=== CatBoost (3000 iters) ===")
    cb = CatBoostRegressor(loss_function='MAE', iterations=3000, learning_rate=0.05,
        depth=8, l2_leaf_reg=3, random_seed=SEED, task_type='CPU', verbose=0)
    oof_cb, pred_cb, _ = run_kfold(cb, X_train, y_train, X_test, 'CatBoost', use_log=True)
else:
    print("=== CatBoost unavailable — LGB substitute ===")
    lgb_s = dict(lgb_p); lgb_s.update({'random_state':SEED+100,'num_leaves':150,'learning_rate':0.035,'n_estimators':3500})
    oof_cb, pred_cb, _ = run_kfold(lgb.LGBMRegressor(**lgb_s), X_train, y_train, X_test, 'LGB-sub', use_log=True)


In [ ]:
# ── MODEL 4: CatBoost deeper (3500 iters, depth=10) ──
if HAS_CATBOOST:
    print("=== CatBoost Deeper (3500 iters, depth=10) ===")
    cb2 = CatBoostRegressor(loss_function='MAE', iterations=3500, learning_rate=0.04,
        depth=10, l2_leaf_reg=5, random_seed=SEED+1, task_type='CPU', verbose=0)
    oof_cb2, pred_cb2, _ = run_kfold(cb2, X_train, y_train, X_test, 'CB-deep', use_log=True)
else:
    lgb_s2 = dict(lgb_p); lgb_s2.update({'random_state':SEED+200,'num_leaves':180,'learning_rate':0.03,'n_estimators':4000})
    oof_cb2, pred_cb2, _ = run_kfold(lgb.LGBMRegressor(**lgb_s2), X_train, y_train, X_test, 'LGB-sub2', use_log=True)


In [ ]:
# ── Optimized blend ──
from scipy.optimize import minimize

oof_stack  = [oof_lgb, oof_lgb2, oof_cb, oof_cb2]
pred_stack = [pred_lgb, pred_lgb2, pred_cb, pred_cb2]
names      = ['LGB','LGB-S2','CB','CB-deep']

def neg_r2_blend(w, oofs, y):
    w = np.clip(w, 0, None); w /= w.sum()
    return -r2_score(y, np.clip(sum(w[i]*oofs[i] for i in range(len(oofs))), 0, None))

res = minimize(neg_r2_blend, x0=[0.25]*4, args=(oof_stack, y_train),
               method='Nelder-Mead', options={'maxiter':5000})
opt_w = np.clip(res.x, 0, None); opt_w /= opt_w.sum()

print("=== Optimal Blend Weights ===")
for n,w in zip(names, opt_w): print(f"  {n:8s}: {w:.4f}")

oof_blend  = sum(opt_w[i]*oof_stack[i]  for i in range(4))
pred_blend = sum(opt_w[i]*pred_stack[i] for i in range(4))
blend_r2   = r2_score(y_train, np.clip(oof_blend, 0, None))
print(f"\nBlend OOF R² = {blend_r2:.5f}  (≈ {100*blend_r2:.2f})")


In [ ]:
# ── Individual scores + submission ──
print("=== OOF R² per model ===")
for n, o in zip(names, oof_stack):
    s = r2_score(y_train, np.clip(o,0,None))
    print(f"  {n:8s}: {s:.5f}  (≈{100*s:.2f})")
print(f"  {'Blend':8s}: {blend_r2:.5f}  (≈{100*blend_r2:.2f})")

final = np.clip(pred_blend, 0, 1)
print(f"\nPrediction stats: min={final.min():.5f} max={final.max():.5f} mean={final.mean():.5f}")
print(f"Train target    : min={y_train.min():.5f} max={y_train.max():.5f} mean={y_train.mean():.5f}")

sub = pd.DataFrame({'Index': test['Index'].astype(int), 'demand': final})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f"\n✅ submission.csv saved → shape={sub.shape}")

In [ ]:
# ── Feature importance ──
lgb_imp = lgb.LGBMRegressor(**lgb_p)
lgb_imp.fit(X_train, np.log1p(y_train))
imp = pd.DataFrame({'feature':FEATURE_COLS,'importance':lgb_imp.feature_importances_})
imp = imp.sort_values('importance', ascending=False).head(30)
print("=== Top-30 Feature Importances ===")
print(imp.to_string(index=False))
